# W01-0 · Regex basics *(before class, ~25 min)*

A **regular expression** is a small pattern language for describing text. This notebook covers the parts you need
for Workshop 1, one idea at a time: run each demo, predict what it will match before you look at the output, then do the
short ✍️ exercise underneath. Replace `...` with your pattern; each exercise checks itself.

W01-A and W01-B in class assume everything here.

> **Tip:** [regex101.com](https://regex101.com/) (flavour: **Python**) explains any pattern piece by piece as you type it.
> It's the fastest way to debug a pattern that doesn't do what you expect.

In [ ]:
import re

def show(pattern, text, flags=0):
    """Print the text with every match in [brackets], then the number of matches."""
    matches = list(re.finditer(pattern, text, flags))
    print(re.sub(pattern, lambda m: f"[{m.group(0)}]", text, flags=flags), f"   ({len(matches)} match{'' if len(matches) == 1 else 'es'})")

show(r"belt", "The dryer belt snapped, so I replaced the belt.")

## 1 · Literal text and the `re` functions

Most characters simply match themselves. The `re` module has one function per job:

| function | returns |
|---|---|
| `re.search(p, s)` | the **first** match as a `Match` object, or `None` |
| `re.findall(p, s)` | a **list** of every match |
| `re.sub(p, repl, s)` | a copy of `s` with every match **replaced** |
| `re.split(p, s)` | `s` split **at** every match |

Matching is case-sensitive unless you pass `flags=re.I` (or start the pattern with `(?i)`).

In [ ]:
text = "The dryer belt snapped. The new belt fits, and the dryer is quiet."

print(re.search(r"belt", text))    # Match object: span=(where it starts, where it ends)
print(re.findall(r"belt", text))
print(re.search(r"motor", text))   # no match

show(r"the", text)                 # case-sensitive: "The" is not matched
show(r"the", text, flags=re.I)

## 2 · Special characters and escaping

Twelve characters have special meanings: `. ^ $ * + ? { } [ ] \ | ( )`.
The most common one is `.`, which matches **any single character** (except a newline).
To match one of these characters literally, put a backslash in front of it: `\.` matches a real dot, `\$` a real dollar sign.

Always write patterns as **raw strings** (`r"..."`), so that Python passes your backslashes through to `re` untouched.

In [ ]:
s = "Paid 24.99 today. Codes 24-99 and 24099 are something else."
show(r"24.99", s)     # . matches any character, so all three match
show(r"24\.99", s)    # \. matches only a real dot

print(len("\b"), len(r"\b"))   # "\b" is Python's backspace character (1 char); r"\b" is backslash + b (2 chars)

## 3 · Character classes: one character from a set

| pattern | matches one character that is… |
|---|---|
| `[aeiou]` | any of the listed characters |
| `[0-9]`, `[A-Z]`, `[a-zA-Z]` | in a range |
| `[^0-9]` | **not** in the set (`^` right after `[` negates it) |
| `\d` · `\D` | a digit · not a digit |
| `\w` · `\W` | a word character (letter, digit or `_`) · anything else |
| `\s` · `\S` | whitespace (space, tab, newline) · anything else |

Inside `[ ]`, most special characters lose their meaning: `[.$]` matches a literal dot or dollar sign.

In [ ]:
s = "Models A7, B12 and C-300 cost ₹499, ₹1,299 and ₹12,000."
show(r"[ABC]", s)
show(r"[0-9]", s)
show(r"\d", s)          # the same as [0-9]
show(r"[^\w\s]", s)     # not a word character and not whitespace: punctuation and symbols

### ✍️ Exercise 1: find the vowels

Match every vowel, **upper or lower case**, in the sentence below. Use a single character class and no flags.

In [ ]:
p1 = r"..."   # ✍️
found = re.findall(p1, "Ice maker Out of order")
print(found)
assert found == ["I", "e", "a", "e", "O", "u", "o", "o", "e"], "not quite: check upper and lower case"
print("✅")

## 4 · Quantifiers: how many times

A quantifier applies to the thing **immediately before it** (one character, one class, or one group).

| quantifier | means |
|---|---|
| `?` | 0 or 1 time (optional) |
| `*` | 0 or more times |
| `+` | 1 or more times |
| `{3}` · `{2,}` · `{2,4}` | exactly 3 · 2 or more · 2 to 4 |

In [ ]:
s = "colour color colouur | 5 55 555 5555"
show(r"colou?r", s)   # the u is optional
show(r"\d+", s)       # one or more digits: each run of digits is one match
show(r"5{2,3}", s)    # 2 to 3 fives; "5555" becomes "555" + a leftover "5" that doesn't match

### Greedy vs lazy

Quantifiers are **greedy**: they match as much as they can while still letting the whole pattern succeed.
Add `?` after the quantifier (`*?`, `+?`) to make it **lazy**: as little as possible.

In [ ]:
html = "<b>Great</b> filter, <i>easy</i> to fit"
show(r"<.+>", html)    # greedy: one match from the first < to the very last >
show(r"<.+?>", html)   # lazy: each tag separately

### ✍️ Exercise 2: times of day

Match the times `9:05`, `10:30` and `23:59`: one or two digits, a colon, then exactly two digits. `3:1` is a ratio, not a time.

In [ ]:
p2 = r"..."   # ✍️
found = re.findall(p2, "Delivered at 9:05, called back at 10:30 and 23:59. Ratio 3:1.")
print(found)
assert found == ["9:05", "10:30", "23:59"], "not quite"
print("✅")

## 5 · Anchors and word boundaries: *where* a match may happen

Anchors match a **position**, not a character.

| anchor | position |
|---|---|
| `^` · `$` | start · end of the text (of each **line** with `flags=re.M`) |
| `\b` | a word boundary: between a word character (`\w`) and a non-word character, or the edge of the text |

In [ ]:
s = "cat concatenate cats scatter cat"
show(r"cat", s)
show(r"\bcat\b", s)    # only the whole word "cat"
show(r"^cat", s)       # only at the start of the text
show(r"cat$", s)       # only at the end

log = "ERROR disk full\nINFO all good\nERROR fan failure"
print(re.findall(r"^ERROR.*", log))               # ^ = start of the whole text, so one match
print(re.findall(r"^ERROR.*", log, flags=re.M))   # re.M: ^ = start of each line

### ✍️ Exercise 3: four-digit years

Match the years `1998` and `2024`, but not the `2019` inside `2019X` or the digits of `20245`.

In [ ]:
p3 = r"..."   # ✍️
found = re.findall(p3, "Since 1998 we sold model 2019X, 20245 units, and a new range in 2024.")
print(found)
assert found == ["1998", "2024"], "not quite: which matches are extra or missing?"
print("✅")

## 6 · Alternation and groups

- `a|b` matches `a` **or** `b`. It has the lowest priority of all, so `^ERROR|WARN` means `(^ERROR)` or `(WARN)`.
- `( … )` groups part of a pattern, so a quantifier or `|` applies to the whole group, **and** it *captures* what it matched.
- `(?: … )` groups without capturing.
- `(?P<name> … )` captures under a name.

In [ ]:
s = "Ordered a fridge, a freezer and a dishwasher."
show(r"fridge|freezer", s)
show(r"fr(?:idge|eezer)", s)   # the group limits the | to the ending

m = re.search(r"(\d+) (days|months)", "It failed after 6 months of use.")
print(m.group(0), "|", m.group(1), "|", m.group(2))   # 0 = the whole match, 1 and 2 = the groups

m = re.search(r"(?P<n>\d+) (?P<unit>days|months)", "It failed after 6 months of use.")
print(m["n"], m["unit"], m.groupdict())

⚠️ **`findall` and groups.** With no groups, `findall` returns the whole matches. With one group it returns **only that group**,
and with several groups it returns **tuples**. Use `(?: )` when you need grouping but want the whole match back.

In [ ]:
s = "6 months, then 10 days"
print(re.findall(r"\d+ (?:days|months)", s))   # no capturing group: whole matches
print(re.findall(r"\d+ (days|months)", s))     # one group: just the group
print(re.findall(r"(\d+) (days|months)", s))   # two groups: tuples

### ✍️ Exercise 4: split emails into name and domain

Use two capturing groups so that `findall` returns `(name, domain)` pairs.

In [ ]:
p4 = r"..."   # ✍️
found = re.findall(p4, "Write to a.rao@gmail.com or support@ghartech.in")
print(found)
assert found == [("a.rao", "gmail.com"), ("support", "ghartech.in")], "not quite: you need exactly two groups"
print("✅")

## 7 · Substitution, backreferences and splitting

In the replacement for `re.sub`, `\1`, `\2`, … insert what each group matched, so you can reorder or reuse parts of the match.

In [ ]:
s = "Delivered 29/09/2026, returned 03/10/2026"
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", s))   # DD/MM/YYYY -> YYYY-MM-DD

print(re.sub(r"\s+", " ", "too    many \t  spaces"))        # collapse whitespace
print(re.split(r"[;,]\s*", "fridge, oven;kettle,  toaster"))   # split on ; or , plus any spaces

### ✍️ Exercise 5: mask a phone number

Replace the first six digits of each 10-digit number with `******` and keep the last four. Use a group and `\1`.

In [ ]:
p5 = r"..."   # ✍️
masked = re.sub(p5, r"******\1", "Call 9822967803 or 7012345678, ticket 12345.")
print(masked)
assert masked == "Call ******7803 or ******5678, ticket 12345.", "not quite"
print("✅")

## 8 · Lookarounds: check the neighbours without matching them

A lookaround tests what comes **before** or **after** the current position, without including it in the match.
W01-B uses these to keep phone numbers from matching inside longer codes.

| pattern | means |
|---|---|
| `(?=…)` · `(?!…)` | followed by … · **not** followed by … |
| `(?<=…)` · `(?<!…)` | preceded by … · **not** preceded by … |

In [ ]:
s = "Pay ₹499 now, ₹1200 later, or 300 points"
show(r"(?<=₹)\d+", s)        # digits preceded by ₹; the ₹ is not part of the match
show(r"\d+(?= points)", s)   # digits followed by " points"

show(r"(?<!\d)\d{3}(?!\d)", "id 12345, code 678, pin 90")   # exactly three digits, not part of a longer number

## 9 · Readable patterns with `re.VERBOSE`

Long patterns are hard to read. With `flags=re.X` (verbose), whitespace in the pattern is ignored and `#` starts a comment,
so you can lay a pattern out one piece per line. To match a real space in verbose mode, use `\s` or `[ ]`.

In [ ]:
price = re.compile(r"""
    (?:₹|Rs\.?)\s?         # currency: ₹, Rs or Rs.
    (\d{1,3}(?:,\d{3})*)   # rupees, with optional thousands commas
    (?:\.(\d{2}))?         # optional paise
""", flags=re.X)

print(price.findall("Now ₹1,299.50, was Rs. 1,499 and Rs899"))

## Cheat sheet

| | |
|---|---|
| **Characters** | `.` any · `\d` digit · `\w` word char · `\s` space · `[abc]` one of · `[^abc]` none of · `\.` literal dot |
| **Quantifiers** | `?` 0–1 · `*` 0+ · `+` 1+ · `{n}` · `{n,}` · `{n,m}` · add `?` for lazy |
| **Positions** | `^` start · `$` end · `\b` word boundary · lookarounds `(?=)` `(?!)` `(?<=)` `(?<!)` |
| **Groups** | `( )` capture · `(?: )` group only · `(?P<name> )` named · `a\|b` or · `\1` in `re.sub` |
| **Flags** | `re.I` ignore case · `re.M` per-line `^` `$` · `re.X` verbose |

➡️ In class, **W01-A** applies all of this to a whole pandas column of 30,000 reviews.